# Notebook 4 – Duplicate Data Handling

Identifying and handling duplicate records in the `customer_transactions_raw.csv` dataset.

## 1. What are Duplicate Records?

Duplicate records are rows in a dataset that repeat information already present elsewhere in the dataset. They can occur due to repeated data entry, system errors, merging multiple data sources, or repeated exports/imports of the same data.

In [1]:
import pandas as pd
df = pd.read_csv('customer_transactions_raw.csv')
df.shape

(1000, 12)

In [2]:
df.head()

,customer_id,age,gender,annual_income,city,membership_type,purchase_amount,quantity,signup_date,payment_method,rating,notes
0,100508,43.0,female,80242.98,Bengaluru,Gold,99.13,1.0,08-22-2021,Debit Card,1,NaN
1,100819,41.0,Female,NaN,Delhi,Silver,113.23,1.0,16 Jun 2020,Net Banking,4,NaN
2,100453,61.0,Male,56285.40,Bengaluru,Gold,248.19,2.0,28 Sep 2020,Debit Card,4,NaN
3,100369,NaN,Male,81878.74,NaN,Silver,51.27,9.0,10 Sep 2023,Credit Card,4,NaN
4,100243,24.0,Male,100566.42,Delhi,Silver,97.97,1.0,07/01/2021,Credit Card,4,NaN


## 2. Exact Duplicates

Exact duplicates are rows where every single column value matches another row exactly.

In [3]:
exact_duplicates = df[df.duplicated(keep=False)]
exact_duplicates.shape

(100, 12)

In [4]:
exact_duplicates.head(10)

,customer_id,age,gender,annual_income,city,membership_type,purchase_amount,quantity,signup_date,payment_method,rating,notes
12,100895,34.0,M,39852.35,NaN,Silver,64.05,5.0,02/12/2022,COD,4,NaN
14,100880,38.0,NaN,61897.39,Bengaluru,Bronze,82.85,1.0,05/02/2020,UPI,1,NaN
38,100645,34.0,male,51420.41,delhi,NaN,348.42,6.0,15/03/2021,NaN,1,NaN
49,100621,33.0,Female,108173.18,bengaluru,NaN,175.14,4.0,12/06/2020,COD,5,NaN
73,100455,39.0,Female,87675.79,BENGALURU,Silver,139.28,4.0,04-06-2019,Debit Card,1,NaN
84,100501,54.0,Male,56335.35,BENGALURU,Silver,72.66,8.0,2020-01-23,Debit Card,4,NaN
85,100500,20.0,Female,55779.47,NaN,Bronze,194.96,7.0,02-24-2023,UPI,5,NaN
93,100160,NaN,MALE,66960.04,Delhi,Silver,286.42,6.0,NaN,Net Banking,3,NaN
101,100014,52.0,Female,49280.47,Delhi,Silver,24.01,5.0,10-11-2023,Net Banking,1,NaN
112,100446,20.0,Male,50100.34,HYDERABAD,Silver,203.67,5.0,12-16-2023,Credit Card,2,NaN


## 3. Partial Duplicates

Partial duplicates are rows that match on some key columns (for example `customer_id`) but differ in other columns. These are trickier since they are not caught by a full-row comparison.

In [5]:
partial_duplicates = df[df.duplicated(subset=['customer_id'], keep=False)]
partial_duplicates.shape

(100, 12)

In [6]:
partial_duplicates.sort_values('customer_id').head(10)

,customer_id,age,gender,annual_income,city,membership_type,purchase_amount,quantity,signup_date,payment_method,rating,notes
101,100014,52.0,Female,49280.47,Delhi,Silver,24.01,5.0,10-11-2023,Net Banking,1,NaN
637,100014,52.0,Female,49280.47,Delhi,Silver,24.01,5.0,10-11-2023,Net Banking,1,NaN
744,100025,33.0,Male,72840.22,NaN,NaN,146.23,4.0,2021-09-19,Net Banking,3,NaN
134,100025,33.0,Male,72840.22,NaN,NaN,146.23,4.0,2021-09-19,Net Banking,3,NaN
201,100026,34.0,F,71076.75,Mumbai,Gold,63.66,2.0,29/05/2021,COD,5,NaN
921,100026,34.0,F,71076.75,Mumbai,Gold,63.66,2.0,29/05/2021,COD,5,NaN
689,100061,18.0,Female,55130.64,mumbai,Gold,94.12,1.0,2020-10-10,Credit Card,3,NaN
219,100061,18.0,Female,55130.64,mumbai,Gold,94.12,1.0,2020-10-10,Credit Card,3,NaN
287,100111,43.0,Female,72741.17,NaN,Platinum,211.08,1.0,31/10/2020,Debit Card,4,NaN
987,100111,43.0,Female,72741.17,NaN,Platinum,211.08,1.0,31/10/2020,Debit Card,4,NaN


## 4. Identifying Duplicates

### 4.1 `duplicated()`

`duplicated()` returns a boolean Series marking rows that are duplicates of a previous row.

In [7]:
df.duplicated().sum()

np.int64(50)

In [8]:
df.duplicated().value_counts()

False    950
True      50
Name: count, dtype: int64

In [9]:
df['is_duplicate'] = df.duplicated()
df['is_duplicate'].sum()

np.int64(50)

### 4.2 `drop_duplicates()`

`drop_duplicates()` removes duplicate rows and returns a cleaned DataFrame.

In [10]:
df_no_exact_dupes = df.drop_duplicates()
df_no_exact_dupes.shape

(1000, 13)

In [11]:
df_no_exact_dupes = df.drop_duplicates(keep='first')
df_no_exact_dupes.shape

(1000, 13)

## 5. Duplicate Detection Based on Selected Columns

Instead of comparing all columns, duplicates can be flagged using a subset of columns that represent a business key.

In [12]:
subset_cols = ['customer_id', 'signup_date']
df.duplicated(subset=subset_cols).sum()

np.int64(50)

In [13]:
df[df.duplicated(subset=['customer_id'], keep=False)].sort_values('customer_id').head(10)

,customer_id,age,gender,annual_income,city,membership_type,purchase_amount,quantity,signup_date,payment_method,rating,notes,is_duplicate
101,100014,52.0,Female,49280.47,Delhi,Silver,24.01,5.0,10-11-2023,Net Banking,1,NaN,False
637,100014,52.0,Female,49280.47,Delhi,Silver,24.01,5.0,10-11-2023,Net Banking,1,NaN,True
744,100025,33.0,Male,72840.22,NaN,NaN,146.23,4.0,2021-09-19,Net Banking,3,NaN,True
134,100025,33.0,Male,72840.22,NaN,NaN,146.23,4.0,2021-09-19,Net Banking,3,NaN,False
201,100026,34.0,F,71076.75,Mumbai,Gold,63.66,2.0,29/05/2021,COD,5,NaN,False
921,100026,34.0,F,71076.75,Mumbai,Gold,63.66,2.0,29/05/2021,COD,5,NaN,True
689,100061,18.0,Female,55130.64,mumbai,Gold,94.12,1.0,2020-10-10,Credit Card,3,NaN,True
219,100061,18.0,Female,55130.64,mumbai,Gold,94.12,1.0,2020-10-10,Credit Card,3,NaN,False
287,100111,43.0,Female,72741.17,NaN,Platinum,211.08,1.0,31/10/2020,Debit Card,4,NaN,False
987,100111,43.0,Female,72741.17,NaN,Platinum,211.08,1.0,31/10/2020,Debit Card,4,NaN,True


## 6. Handling Duplicate Records

Once duplicates are identified, they can be handled in several ways: dropping them, keeping the first or last occurrence, aggregating them, or flagging them for manual review.

In [14]:
df_clean = df.drop_duplicates(subset=['customer_id'], keep='last')
df_clean.shape

(950, 13)

In [15]:
df_clean = df.drop_duplicates(subset=['customer_id'], keep='first')
df_clean.shape

(950, 13)

In [16]:
agg_df = df.groupby('customer_id').agg({'purchase_amount': 'sum', 'quantity': 'sum'}).reset_index()
agg_df.head()

,customer_id,purchase_amount,quantity
0,100001,28.79,8.0
1,100002,37.2,9.0
2,100003,19.83,9.0
3,100004,11.05,6.0
4,100005,224.78,7.0


## 7. Business Rules for Duplicate Removal

Duplicate removal should be guided by business context rather than blind automation. Some common rules:

- Keep the record with the most recent `signup_date` or transaction date.
- Keep the record with the most complete information (fewest missing values).
- Keep the record with the highest `rating` or `purchase_amount` if it reflects a corrected entry.
- Aggregate quantities/amounts instead of dropping when duplicates represent multiple valid transactions by the same customer.

In [17]:
df['missing_count'] = df.isnull().sum(axis=1)
df_sorted = df.sort_values(['customer_id', 'missing_count'])
df_business_clean = df_sorted.drop_duplicates(subset=['customer_id'], keep='first')
df_business_clean.shape

(950, 14)

In [18]:
df_business_clean = df_business_clean.drop(columns=['is_duplicate', 'missing_count'])
df_business_clean.to_csv('customer_transactions_deduped.csv', index=False)
df_business_clean.shape

(950, 12)

## 8. Why Blindly Deleting Duplicates Can Be Dangerous

Removing duplicates without understanding the business context can introduce serious errors:

- **Legitimate repeat transactions look like duplicates.** A customer buying the same product twice in one day produces two rows that are nearly identical but both real. Dropping one loses genuine revenue and quantity data.
- **Row-level duplication hides in partial matches.** Two different customers can share the same non-key attributes (age, city, membership type) by coincidence. Deduplicating on the wrong subset of columns can wrongly merge unrelated customers.
- **`keep='first'` or `keep='last'` is an arbitrary choice.** Without checking which row is more accurate or more recent, the retained row may actually be the incorrect or outdated one.
- **Aggregated information gets lost.** If duplicates represent separate purchases, summing or averaging is often correct; dropping rows outright silently discards that information (e.g. total revenue, total quantity sold).
- **Downstream metrics get skewed.** Deleting rows changes counts, sums, and averages used in reporting, dashboards, and machine learning features, which can distort business decisions.
- **Data lineage and auditability suffer.** In domains like finance and healthcare, removing rows without a documented rule can violate compliance and audit requirements, since a record's disappearance must be explainable.

The safe approach is to first understand *why* duplicates exist, define explicit business rules for which record to keep or how to combine them, and only then apply deduplication — ideally logging what was removed so the process is reversible and auditable.